# 11 - PySpark Forecast vs Actual


## Setup
Target: Configure Spark + JDBC helper.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
spark = (SparkSession.builder.appName('capacity-demo').config('spark.driver.host','127.0.0.1').config('spark.driver.bindAddress','127.0.0.1').config('spark.jars.packages','org.postgresql:postgresql:42.7.4').getOrCreate())
JDBC_URL = 'jdbc:postgresql://host.docker.internal:5432/observability'
def load_query(q: str):
    return (spark.read.format('jdbc').option('url', JDBC_URL).option('dbtable', f'({q}) t').option('user','obs_user').option('password','obs_pass').option('driver','org.postgresql.Driver').load())


## Forecast Validation
Target: Compare lag prediction with actuals and compute errors.


In [ ]:
q = """SELECT sampled_at, host, cpu_pct, region AS application, env AS service FROM lab.telemetry_cpu_raw WHERE sampled_at >= now() - interval '30 days'"""
df = load_query(q)
daily = df.withColumn('day_bucket', F.to_date('sampled_at')).groupBy('day_bucket','host','application','service').agg(F.max('cpu_pct').alias('actual_peak_cpu'))
w = Window.partitionBy('host','application','service').orderBy('day_bucket')
out = daily.withColumn('predicted_peak_cpu', F.lag('actual_peak_cpu', 1).over(w)).where(F.col('predicted_peak_cpu').isNotNull()).withColumn('forecast_error', F.col('actual_peak_cpu') - F.col('predicted_peak_cpu')).withColumn('absolute_error', F.abs(F.col('forecast_error')))
out.orderBy(F.col('day_bucket').desc(), 'host').show(40, truncate=False)
